In [ ]:
#XGBoost
from xgboost import XGBRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 4,
    random_state = 42
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_xgboost.csv', index=False)


In [ ]:
#LGBMBoost
from lightgbm import LGBMRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = LGBMRegressor(
        n_estimators = 200,
        learning_rate = 0.05,
        num_leaves = 15,
        max_depth = 4,
        random_state = 42,
        verbose = -1
    )

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lgbmboost.csv', index=False)

In [ ]:
#CatBoost
from catboost import CatBoostRegressor
import pandas as pd
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

fold_rmse = []
    
model = CatBoostRegressor(
        iterations = 200,
        learning_rate = 0.05,
        depth = 4,
        verbose = 0,
        random_state = 42
    )

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_catboost.csv', index=False)

In [ ]:
#ElasticNet (best params)
from sklearn.linear_model import ElasticNet
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        ElasticNet(alpha = 0.01, l1_ratio = 0.5, max_iter = 20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_elasticnet.csv', index=False)


In [ ]:
#CatBoost (best params)
from catboost import CatBoostRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        CatBoostRegressor(iterations = 500, depth = 6, learning_rate = 0.1, verbose = 0, random_state = 42),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_catboost.csv', index=False)


In [ ]:
#XGBoost (best params)
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        XGBRegressor(n_estimators = 500, learning_rate = 0.1, max_depth = 3, colsample_bytree = 0.8, subsample = 0.8, random_state = 42, objective = 'reg:squarederror', eval_metric = 'rmse', n_jobs = 1),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_xgboost.csv', index=False)


In [ ]:
#Lasso (best params)
from sklearn.linear_model import Lasso
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        Lasso(alpha = 0.01, random_state = 42, max_iter=20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lasso.csv', index=False)


In [ ]:
#LGBMBoost (best params)
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        LGBMRegressor(n_estimators = 500, learning_rate = 0.05, max_depth = 3, num_leaves = 7, random_state=42, verbose=-1, n_jobs=1),
        preprocessor,
        scale = False
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_lgbmboost.csv', index=False)


In [ ]:
#Ridge (best params)
from sklearn.linear_model import Ridge
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

model = TransformedTargetRegressor(
    regressor = create_pipe(
        Ridge(alpha = 100, random_state = 42, max_iter=20000),
        preprocessor,
        scale = True
    ),
    func = np.log1p,
    inverse_func = np.expm1
)

model.fit(X, y)
predictions = model.predict(X_test)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
submission.to_csv('submission_ridge.csv', index=False)

In [ ]:
#best voting
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'Lasso': create_model(
        Lasso(
            alpha=0.01,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'Ridge': create_model(
        Ridge(
            alpha=100,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    )
}

voting_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('elasticnet', best_estimators['ElasticNet']),
        ('xgb', best_estimators['XGBoost']),
        ('lasso', best_estimators['Lasso']),
        ('ridge', best_estimators['Ridge'])
    ]
)

voting_model.fit(X, y)

predictions = voting_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/voting_submission.csv', index=False)

In [ ]:
#best stacking ridge
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'Lasso': create_model(
        Lasso(
            alpha=0.01,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'Ridge': create_model(
        Ridge(
            alpha=100,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    )
}

stacking_model = StackingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('elasticnet', best_estimators['ElasticNet']),
        ('xgb', best_estimators['XGBoost']),
        ('lasso', best_estimators['Lasso']),
        ('ridge', best_estimators['Ridge'])
    ],
    final_estimator=Ridge(),
    cv=5,
    n_jobs=1
)

stacking_model.fit(X, y)

predictions = stacking_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/stacking_ridge_submission.csv', index=False)

In [ ]:
#best Bagging elasticnet
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),
}

bagging_model = create_model(
    BaggingRegressor(
        estimator=ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        n_estimators=50,
        random_state=42,
        n_jobs=1
    ),
    preprocessor,
    scale=True
)

bagging_model.fit(X, y)

predictions = bagging_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/bagging_elastic_submission.csv', index=False)

In [ ]:
#best weighted aver
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),
}

weighted_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('xgb', best_estimators['XGBoost'])
    ],
    weights=[2, 3, 2]
)

weighted_model.fit(X, y)

predictions = weighted_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/weighted_aver_submission.csv', index=False)

In [ ]:
#best Bagging ridge
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'Ridge': create_model(
        Ridge(
            alpha=100,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),
}

bagging_model = create_model(
    BaggingRegressor(
        estimator=Ridge(
            alpha=100,
            max_iter=20000
        ),
        n_estimators=50,
        random_state=42,
        n_jobs=1
    ),
    preprocessor,
    scale=True
)

bagging_model.fit(X, y)

predictions = bagging_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/bagging_ridge_submission.csv', index=False)

In [ ]:
#best Stohastic grad boost
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

sgb_model = create_model(
    GradientBoostingRegressor(
        n_estimators = 100,
        learning_rate=0.05,
        max_depth=3,
        subsample = 0.8,
        max_features = 'sqrt',
        random_state = 42
    ),
    preprocessor,
    scale = False
)

sgb_model.fit(X, y)

predictions = sgb_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/sgb_model_submission.csv', index=False)

In [ ]:
#best voting
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    )
}

voting_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('elasticnet', best_estimators['ElasticNet']),
        ('xgb', best_estimators['XGBoost'])
    ]
)

voting_model.fit(X, y)

predictions = voting_model.predict(X_test)
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})
submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/voting_submission.csv', index=False)

In [4]:
#Blending weight aver + voting
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_log_error
import sys
sys.path.append(r'C:\Users\User\house-prices-competitions')
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    )
}


voting_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('elasticnet', best_estimators['ElasticNet']),
        ('xgb', best_estimators['XGBoost'])
    ]
)

weighted_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('xgb', best_estimators['XGBoost'])
    ],
    weights=[2, 3, 2]
)

cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

oof_voting = cross_val_predict(
    voting_model,
    X,
    y,
    cv = cv,
    n_jobs = 1
)

oof_weighted = cross_val_predict(
    weighted_model,
    X,
    y,
    cv = cv,
    n_jobs = 1
)

def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_pred = np.clip(y_pred, 0, None)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

weights = [0.25, 0.4, 0.5, 0.6, 0.75]
blend_res = []
for i in weights:
    blend_pred = (i * oof_voting) + (1 - i) * oof_weighted
    score = rmsle(y, blend_pred)
    blend_res.append({
        'Voting': i,
        'Weighted': 1 - i,
        'RMSLE': score
    })
    print(
        f'Voting: {i:.0%} | '
        f'Weighted: {1-i:.0%} | '
        f'RMSLE: {score:.6f}'
    )

blend_results = pd.DataFrame(blend_res)

blend_results = blend_results.sort_values(
    by='RMSLE'
).reset_index(drop=True)

print('\nРезультаты эксперимента:')
print(blend_results.to_string(index=False))

best_blend = blend_results.iloc[0]

print('\nЛучший вариант blending:')
print(f"Voting: {best_blend['Voting']:.0%}")
print(f"Weighted Averaging: {best_blend['Weighted']:.0%}")
print(f"OOF RMSLE: {best_blend['RMSLE']:.6f}")

Voting: 25% | Weighted: 75% | RMSLE: 0.112286
Voting: 40% | Weighted: 60% | RMSLE: 0.111818
Voting: 50% | Weighted: 50% | RMSLE: 0.111528
Voting: 60% | Weighted: 40% | RMSLE: 0.111256
Voting: 75% | Weighted: 25% | RMSLE: 0.110880

Результаты эксперимента:
 Voting  Weighted    RMSLE
   0.75      0.25 0.110880
   0.60      0.40 0.111256
   0.50      0.50 0.111528
   0.40      0.60 0.111818
   0.25      0.75 0.112286

Лучший вариант blending:
Voting: 75%
Weighted Averaging: 25%
OOF RMSLE: 0.110880


In [ ]:
#Blending weight aver + voting
import os
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_log_error
import sys
sys.path.append(r'C:\Users\User\house-prices-competitions')
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    )
}


voting_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('elasticnet', best_estimators['ElasticNet']),
        ('xgb', best_estimators['XGBoost'])
    ]
)

weighted_model = VotingRegressor(
    estimators=[
        ('lgbm', best_estimators['LightGBM']),
        ('catboost', best_estimators['CatBoost']),
        ('xgb', best_estimators['XGBoost'])
    ],
    weights=[2, 3, 2]
)

cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

oof_voting = cross_val_predict(
    voting_model,
    X,
    y,
    cv = cv,
    n_jobs = 1
)

oof_weighted = cross_val_predict(
    weighted_model,
    X,
    y,
    cv = cv,
    n_jobs = 1
)

def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_pred = np.clip(y_pred, 0, None)

    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

best_voting_weight = 0.75
best_weighted_weight = 0.25
blend_pred = best_voting_weight * oof_voting + best_weighted_weight * oof_weighted
score = rmsle(y, blend_pred)
    
voting_model.fit(X, y)
weighted_model.fit(X, y)
test_voting = voting_model.predict(X_test)
test_weighted = weighted_model.predict(X_test)

final_pred = 0.75 * test_voting + 0.25 * test_weighted
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_pred
})

submission['SalePrice'] = submission['SalePrice'].clip(lower=0)
os.makedirs('submissions', exist_ok=True)
submission.to_csv('submissions/blend_vot_weight.csv', index=False)